# Chapter 2 · Examples and step-by-step reasoning

An e-mail classifier with and without examples in the instruction, measured with accuracy and the confusion matrix, and refund calculations answering directly, reasoning step by step, with a worked example and voting among several answers.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*. With a GPU, the notebook uses Qwen2.5-1.5B-Instruct; without one, Qwen3-0.6B. The classification takes about a minute; the refunds take much longer, because they are seventy reasoned answers.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. The model and the data

The first time, the model and the file with the labeled e-mails are downloaded from the book's repository.

In [ ]:
"""Chapter 2: examples and step-by-step reasoning. Open model, on GPU or CPU."""
import json
import re
import time
import unicodedata
import urllib.request
from collections import Counter
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_GPU = "Qwen/Qwen2.5-1.5B-Instruct"  # with Colab's free T4 GPU
MODEL_CPU = "Qwen/Qwen3-0.6B"             # without a GPU: smaller and faster
MODEL = None                              # None: choose according to the machine
SEED = 0
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/en/"



def load(name=None):
    """Loads the model and its tokenizer on the GPU if there is one; if not, on CPU."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if name is None:
        name = MODEL_GPU if device == "cuda" else MODEL_CPU
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
    model.to(device).eval()
    print(f"Model {name} on {device}")
    return tok, model


def load_emails(name="classified_emails.json"):
    """The labeled e-mails; in Colab, they are downloaded from the repository."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return json.loads(path.read_text(encoding="utf-8"))

In [ ]:
tok, model = load(MODEL)
data = load_emails()
print(len(data["examples"]), "examples and", len(data["test"]), "test e-mails")
for category, definition in data["categories"].items():
    print(f"{category}: {definition}")

## 2. Asking with examples

`messages` adds the examples as earlier turns of the conversation.

In [ ]:
def messages(system, user, examples=()):
    """System message, worked examples (question-answer pairs) and the question."""
    conversation = [{"role": "system", "content": system}] if system else []
    for question, answer in examples:
        conversation.append({"role": "user", "content": question})
        conversation.append({"role": "assistant", "content": answer})
    return conversation + [{"role": "user", "content": user}]


def ask(system, user, examples=(), temperature=0, max_tokens=200):
    """Sends a conversation to the model and returns its answer as text."""
    text = tok.apply_chat_template(messages(system, user, examples),
                                   tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False)
    inputs = tok(text, return_tensors="pt").to(model.device)
    if temperature > 0:
        sampling = dict(do_sample=True, temperature=temperature, top_p=1.0, top_k=0)
    else:
        sampling = dict(do_sample=False, temperature=None, top_p=None, top_k=None)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                pad_token_id=tok.eos_token_id, **sampling)
    new = output[0, inputs["input_ids"].shape[1]:]
    answer = tok.decode(new, skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", answer, flags=re.S).strip()

In [ ]:
for m in messages("Classify the e-mail.", "Are you open on Saturdays?", data["examples"][:2]):
    print(m["role"], "→", m["content"])

## 3. Classifying e-mails

In [ ]:
def strip_accents(text):
    """Lowercase and without accents, to compare without worrying about them."""
    return "".join(c for c in unicodedata.normalize("NFD", text.lower())
                   if unicodedata.category(c) != "Mn")


def read_category(answer, categories):
    """The first category that appears in the answer, or None if there is none."""
    text = strip_accents(answer)
    found = {c: re.search(c[:-1] + "(y|ies)" if c.endswith("y") else c, text)
             for c in categories}                # inquiry or inquiries
    positions = {c: m.start() for c, m in found.items() if m}
    return min(positions, key=positions.get) if positions else None


def instruction(data, definitions):
    """The system message: the task and the categories, with or without definitions."""
    lines = [f"- {c}: {d}" if definitions else f"- {c}"
             for c, d in data["categories"].items()]
    return ("Classify an e-mail from a customer of North Cycles, a bicycle store, "
            "into one of these categories:\n" + "\n".join(lines)
            + "\nReply with the category name only.")


def classify(data, definitions=True, examples=()):
    """Classifies the test e-mails and returns the (correct, predicted) pairs."""
    system = instruction(data, definitions)
    pairs = []
    for email, correct in data["test"]:
        r = ask(system, email, examples, max_tokens=10)
        pairs.append((correct, read_category(r, data["categories"])))
    return pairs


def accuracy(pairs):
    """Proportion of e-mails classified correctly."""
    return sum(c == p for c, p in pairs) / len(pairs)


def confusion_matrix(pairs, categories):
    """Rows: correct category; columns: predicted (the last one, "none")."""
    columns = list(categories) + [None]
    matrix = [[0] * len(columns) for _ in categories]
    for correct, predicted in pairs:
        matrix[list(categories).index(correct)][columns.index(predicted)] += 1
    return matrix

In [ ]:
print(instruction(data, definitions=True))

In [ ]:
def test_classifier(data):
    """Accuracy of the classifier with four versions of the instruction."""
    examples = data["examples"]
    one_per_category = examples[::2]           # the first example of each category
    variants = {
        "Categories only": dict(definitions=False),
        "With definitions": dict(definitions=True),
        "With 5 examples": dict(definitions=True, examples=one_per_category),
        "With 10 examples": dict(definitions=True, examples=examples),
    }
    results = {}
    for name, options in variants.items():
        start = time.time()
        pairs = classify(data, **options)
        results[name] = pairs
        print(f"{name:22s} accuracy {accuracy(pairs):5.1%}  "
              f"({time.time() - start:.0f} s)")
    return results


def show_matrix(pairs, categories):
    """Prints the confusion matrix."""
    matrix = confusion_matrix(pairs, categories)
    print(" " * 11 + "".join(f"{c[:8]:>9s}" for c in categories) + "     none")
    for category, row in zip(categories, matrix):
        print(f"{category:>11s}" + "".join(f"{n:>9d}" for n in row))

In [ ]:
classification = test_classifier(data)

In [ ]:
show_matrix(classification["Categories only"], data["categories"])

# The e-mails misclassified with each instruction
for variant, pairs in classification.items():
    print("\n==", variant)
    for (correct, predicted), (email, _) in zip(pairs, data["test"]):
        if correct != predicted:
            print(f"{correct} → {predicted}: {email}")

## 4. Calculating refunds

Ten problems with their correct answer and four ways of asking for the calculation.

In [ ]:
PROBLEMS = [
    ("I bought a $120 wheelset and a $30 pump. I'm returning the pump, and you "
     "charge me $5 for the pickup. How much will you refund me?", 25),
    ("I bought an $80 helmet with a 25% discount. I'm returning it. How much "
     "will you refund me?", 60),
    ("I bought two jerseys at $35 each and a pair of $90 shoes, with a 10% "
     "discount on the whole order. I'm returning both jerseys. How much will "
     "you refund me?", 63),
    ("I bought three inner tubes at $8 each and a $46 saddle. I'm returning the "
     "saddle and one inner tube. How much will you refund me?", 54),
    ("I bought a $640 bike and a $40 lock, with a 10% discount on everything. "
     "I'm returning the lock. How much will you refund me?", 36),
    ("I bought 4 water bottles at $7 each. I'm returning half of them. How much "
     "will you refund me?", 14),
    ("I bought a $1,500 e-bike and paid $15 for shipping. It arrived broken, so "
     "I'm returning it and you're refunding the shipping too. How much will you "
     "refund me?", 1515),
    ("I bought a pair of $24 gloves and a pair of $56 sunglasses, with a 25% "
     "discount on everything. I'm returning the sunglasses. How much will you "
     "refund me?", 42),
    ("I bought a $50 rack and a pair of $150 panniers, and I used a $40 coupon "
     "that is split between the products in proportion to their price. I'm "
     "returning the panniers. How much will you refund me?", 120),
    ("I bought 4 pairs of socks at $10 and a pair of $50 bib shorts, with a 10% "
     "discount on everything. I'm returning 2 pairs of socks and the bib "
     "shorts. How much will you refund me?", 63),
]

In [ ]:
REFUND_SYSTEM = ("You are the North Cycles assistant. Work out how much money "
                 "has to be refunded to the customer.")


DIRECT = "Reply only with the amount in dollars, without explanations."


STEP_BY_STEP = ("Reason step by step and end with a final line in the format "
                "\"Answer: $<amount>\".")


WORKED_EXAMPLE = [(
    "I bought a $40 saddle and a $20 light, with a 20% discount on everything. "
    "I'm returning the light. How much will you refund me?",
    "The light cost $20. With the 20% discount, you paid 20 × 0.80 = $16 for "
    "it, and that is what gets refunded.\nAnswer: $16")]

In [ ]:
def read_amount(answer):
    """The amount after "Answer:" or, if there is none, the last number."""
    part = answer.rsplit("Answer:", 1)[-1]
    numbers = re.findall(r"\d+(?:[.,]\d+)*", part)
    if not numbers:
        return None
    number = numbers[0] if "Answer:" in answer else numbers[-1]
    if re.fullmatch(r"\d{1,3}(,\d{3})+(\.\d+)?", number):    # 1,515.50: comma
        number = number.replace(",", "")                    # separates thousands
    return float(number.replace(",", "."))


def refund(problem, style, examples=(), temperature=0):
    """Asks for the calculation in one answer style; returns the text and amount."""
    answer = ask(REFUND_SYSTEM + " " + style, problem, examples,
                 temperature=temperature, max_tokens=400)
    return answer, read_amount(answer)


def vote(problem, times=5):
    """Self-consistency: several sampled answers; the most repeated one wins."""
    amounts = [refund(problem, STEP_BY_STEP, WORKED_EXAMPLE,
                      temperature=0.7)[1] for _ in range(times)]
    valid = [a for a in amounts if a is not None]
    winner = Counter(valid).most_common(1)[0][0] if valid else None
    return winner, amounts


def test_refunds(times=5):
    """The ten problems with the four ways of asking for the answer."""
    torch.manual_seed(SEED)
    rows = []
    for problem, correct in PROBLEMS:
        row = {"problem": problem, "correct": correct}
        for key, style, examples in (("direct", DIRECT, ()),
                                     ("steps", STEP_BY_STEP, ()),
                                     ("example", STEP_BY_STEP, WORKED_EXAMPLE)):
            row["text_" + key], row[key] = refund(problem, style, examples)
        row["vote"], row["votes"] = vote(problem, times)
        rows.append(row)
        print(f"{correct:>6} | direct {row['direct']} | steps {row['steps']}"
              f" | with example {row['example']} | vote {row['vote']}",
              row["votes"])
    for key in ("direct", "steps", "example", "vote"):
        hits = sum(r[key] == r["correct"] for r in rows)
        print(f"Correct {key}: {hits} of {len(rows)}")
    return rows

Before running the ten problems, look at how the model reasons on one of them, without the example and with it:

In [ ]:
problem = PROBLEMS[1][0]
print(refund(problem, STEP_BY_STEP)[0])
print("\n--- With the worked example:\n")
print(refund(problem, STEP_BY_STEP, WORKED_EXAMPLE)[0])

Now, the ten problems in the four ways. This is the slow part of the notebook.

In [ ]:
refunds = test_refunds()

## Exercises

1. Explain in your own words why using examples in the instruction is called "learning," even though the model does not change. How is it similar to training, and how is it different?

2. In the file, the ten examples are grouped by category: first the two orders, then the two returns, and so on. Shuffle them with `random.shuffle` and measure the accuracy again. Does it change? And if you put them in reverse order?

3. Modify `classify` so that, if the answer is none of the categories, it asks again, reminding the model of the list of allowed categories. Do the made-up answers with five examples disappear?

4. Write ten harder test e-mails: long ones, with spelling mistakes, mixing two topics or written with irony. How much does the accuracy of the best instruction drop? Which categories get confused the most?

5. One of the test e-mails, "Can I pay for an e-bike in installments?", confused the model when it had the definitions. Do you think the definition of *inquiry* is clear? Improve it and check whether the result changes.

6. With `messages` and the tokenizer, count how many tokens the ten examples add to each request. If you classified 10,000 e-mails a month with an API that charges $0.50 per million input tokens, how much would the examples cost?

7. Write a second worked example for the refunds, with a discount coupon split in proportion to price, and add it to `WORKED_EXAMPLE`. Does the panniers problem improve? And the others?

8. Repeat the vote with 11 attempts instead of 5. Do the correct answers increase? How long does it take?

9. Write a function `refund_amount(prices, discount, returned)` that does the calculation in Python. Then ask the model to extract the prices, the discount and the returned products from each problem, instead of calculating. How many problems does it solve this way? Compare with chain of thought.

10. If you have a GPU, load Qwen3-0.6B, change `enable_thinking=False` to `True` in `ask` and raise `max_tokens` to 1500. Solve the ten refunds with `DIRECT`. How many does it get right now? How long does each answer take? Read some of its reasoning.

## References

Models [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) and [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), by Qwen (Alibaba), under the Apache 2.0 license. Few-shot learning: [Brown et al. (2020)](https://arxiv.org/abs/2005.14165). Chain of thought: [Wei et al. (2022)](https://arxiv.org/abs/2201.11903) and [Kojima et al. (2022)](https://arxiv.org/abs/2205.11916). Self-consistency: [Wang et al. (2022)](https://arxiv.org/abs/2203.11171). The North Cycles store, its e-mails and the problems are fictional. Original code and experiments.